# PyTorch Basics
Building and training neural networks using PyTorch — understanding tensors, autograd, nn.Module, and the training loop.

## 1. Theory & Intuition

### Why PyTorch over NumPy?
Two things NumPy cannot do:
- **Autograd** — automatic differentiation. PyTorch tracks every operation and computes gradients automatically. No manual backprop needed.
- **GPU acceleration** — run the same code on GPU with one line (`tensor.cuda()`), 10-100x faster for large models.

### Core PyTorch Concepts
- **Tensor** — multi-dimensional array with autograd support (like numpy array but smarter)
- **requires_grad** — tells PyTorch to track gradients for this tensor
- **loss.backward()** — automatically computes all gradients via chain rule
- **optimizer.zero_grad()** — clears gradients before each step (they accumulate by default)
- **optimizer.step()** — updates weights using computed gradients

### The Training Loop (4 steps every time):
1. Forward pass: `output = model(X)`
2. Compute loss: `loss = criterion(output, y)`
3. Backward pass: `loss.backward()`
4. Update weights: `optimizer.step()`

## 2. From Scratch — Tensors & Autograd

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# Basic tensor operations
x = torch.tensor([1.0, 2.0, 3.0])
y = torch.tensor([4.0, 5.0, 6.0])

print("Addition:", x + y)
print("Multiplication:", x * y)
print("Shape:", x.shape)
print("Dtype:", x.dtype)

### Autograd — Automatic Differentiation

In [ ]:
# Autograd example
# y = x² + 3x + 1 → dy/dx = 2x + 3 → at x=2: dy/dx = 7
x = torch.tensor(2.0, requires_grad=True)
y = x ** 2 + 3 * x + 1

y.backward()  # compute gradients automatically
print(f"x = {x.item()}")
print(f"y = {y.item()}")
print(f"dy/dx = {x.grad.item()}")  # should be 7

## 3. PyTorch Implementation — Neural Network

In [ ]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(3, 4)   # 3 inputs → 4 neurons
        self.layer2 = nn.Linear(4, 1)   # 4 neurons → 1 output
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.relu(self.layer1(x))
        x = self.sigmoid(self.layer2(x))
        return x

model = NeuralNetwork()
print(model)

# Test forward pass
X = torch.randn(5, 3)
output = model(X)
print("\nOutput shape:", output.shape)
print("Output values:", output)

## 4. Train & Evaluate

In [ ]:
# Data
X = torch.randn(5, 3)
y = torch.tensor([[1.0], [0.0], [1.0], [0.0], [1.0]])

# Loss and optimizer
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# Training loop
for epoch in range(1000):
    optimizer.zero_grad()       # 1. clear gradients
    output = model(X)           # 2. forward pass
    loss = criterion(output, y) # 3. compute loss
    loss.backward()             # 4. backprop (automatic!)
    optimizer.step()            # 5. update weights
    
    if epoch % 100 == 0:
        print(f"Epoch {epoch}: Loss = {loss.item():.4f}")

# Final evaluation
with torch.no_grad():
    final_output = model(X)
    predictions = (final_output >= 0.5).float()
    accuracy = (predictions == y).float().mean()
    print(f"\nFinal Loss: {loss.item():.4f}")
    print(f"Accuracy: {accuracy.item():.2%}")

## 5. Key Takeaways

### NumPy vs PyTorch Comparison
| | NumPy From Scratch | PyTorch |
|--|---|---|
| Forward pass | Manual matrix ops | `model(X)` |
| Loss | Manual BCE formula | `nn.BCELoss()` |
| Backprop | Manual `backward_pass()` | `loss.backward()` |
| Weight update | Manual `W -= lr * dW` | `optimizer.step()` |
| Final loss | ~0.13 | ~0.007 |

### Key Points
1. `requires_grad=True` tells PyTorch to track gradients for a tensor
2. `loss.backward()` replaces your entire manual backward_pass function
3. `optimizer.zero_grad()` must be called before each step — gradients accumulate by default
4. Adam optimizer converges faster than plain SGD — adapts learning rate per parameter
5. `with torch.no_grad():` disables gradient tracking during inference (saves memory)